# Template: Financial Data from SEC EDGAR API
This notebook downloads financial statement data from the SEC's free EDGAR XBRL API. No registration or API key needed — just an email address.

**What it does:**
1. Maps your tickers to SEC CIK codes
2. Downloads financial facts from the CompanyFacts API (one request per company)
3. Extracts both **annual** and **quarterly** data for every indicator you selected
4. Handles the fact that companies use different XBRL tags for the same concept — and switch tags over time
5. Fills missing quarterly income statement data (the fiscal year-end quarter is not reported separately by EDGAR — we compute it as Annual minus the other 3 quarters)
6. Downloads an Excel file with two tabs: **Annual** and **Quarterly**

**Why EDGAR instead of Yahoo Finance?** Yahoo Finance provides only ~4 quarters of financial statements. EDGAR gives you the full filing history going back 10+ years, which is needed for panel regressions.

In [ ]:
# Cell 1: Install and import libraries
%%capture
!pip install openpyxl

import requests
import pandas as pd
import numpy as np
import re
import time

In [ ]:
# Cell 2: Configuration — EDIT THIS CELL
# ============================================================

# Your email (required by SEC policy — no registration, just identification)
EMAIL = "pavlo.illiashenko@taltech.ee"

# S&P 500 tickers from the shared Google Sheet
SP500_URL = "https://docs.google.com/spreadsheets/d/1o2MUMjnTP2fbi1-GijQDf82IGC5kTDIp/export?format=csv"
sp500 = pd.read_csv(SP500_URL)
TICKERS = sp500['Ticker'].tolist()

# Time period (calendar years)
YEAR_START = 2015
YEAR_END   = 2025

# Indicators to fetch — comment out any you do not need.
# Both annual and quarterly data are fetched automatically.
INDICATORS = [
    # --- Income Statement ---
    'Revenue',               # Total revenue / net sales
    'COGS',                  # Cost of goods sold / cost of revenue
    'GrossProfit',           # Gross profit
    'OperatingIncome',       # Operating income (EBIT)
    'NetIncome',             # Net income (loss)
    'DepreciationAmort',     # Depreciation & amortization
    'InterestExpense',       # Interest expense
    'ResearchDev',           # Research & development expense
    'OperatingExpenses',     # Total operating expenses
    # --- Balance Sheet ---
    'TotalAssets',           # Total assets
    'TotalLiabilities',      # Total liabilities
    'TotalEquity',           # Stockholders' equity
    'Cash',                  # Cash and cash equivalents
    'LongTermDebt',          # Long-term debt (non-current)
    'CurrentDebt',           # Short-term / current debt
    'Inventory',             # Inventory (net)
    'AccountsReceivable',    # Accounts receivable (net, current)
    'AccountsPayable',       # Accounts payable (current)
    'NetPPE',                # Property, plant & equipment (net)
    'CurrentAssets',         # Total current assets
    'CurrentLiabilities',    # Total current liabilities
    # --- Shares ---
    'SharesOutstanding',     # Common shares outstanding
]

# ============================================================

print(f"Email:      {EMAIL}")
print(f"Tickers:    {len(TICKERS)} loaded from Google Sheet")
print(f"Period:     {YEAR_START}–{YEAR_END}")
print(f"Indicators: {len(INDICATORS)} selected")

Email:      pavlo.illiashenko@taltech.ee
Tickers:    500 loaded from Google Sheet
Period:     2015–2025
Indicators: 22 selected


---
## From here, just run all cells (Runtime → Run all)
No need to edit anything below.

In [ ]:
# Cell 3: XBRL tag catalog
#
# Companies use different XBRL tags for the same concept, and switch
# tags over time (e.g., Revenue changed after ASC 606 in 2018).
# This catalog maps each indicator to all known tag variants.
#
# The SEC's 'frame' field handles deduplication:
#   Annual duration (income stmt):  CY2020    = calendar year 2020
#   Annual instant  (balance sheet): CY2020Q4I = year-end 2020 snapshot
#   Quarterly duration:  CY2020Q1   = Q1 2020
#   Quarterly instant:   CY2020Q1I  = end of Q1 2020
#
# Some concepts cross the duration/instant boundary (e.g., SharesOutstanding
# can be instant or weighted-average duration), so the code tries BOTH
# frame patterns for every tag. No false matches occur because the SEC
# assigns frame formats based on the accounting concept type.
#
# 'is_flow' marks income statement items (additive across quarters).
# For these items, the fiscal year-end quarter is often missing from
# EDGAR's quarterly frames — we fill it as Annual minus the other 3 Qs.

XBRL_CATALOG = {
    # --- Income Statement (duration, is_flow=True) ---
    'Revenue':          {'unit': 'USD', 'is_flow': True, 'tags': ['Revenues', 'RevenueFromContractWithCustomerExcludingAssessedTax', 'RevenueFromContractWithCustomerIncludingAssessedTax', 'SalesRevenueNet', 'SalesRevenueGoodsNet', 'SalesRevenueServicesNet']},
    'COGS':             {'unit': 'USD', 'is_flow': True, 'tags': ['CostOfGoodsAndServicesSold', 'CostOfRevenue', 'CostOfGoodsSold']},
    'GrossProfit':      {'unit': 'USD', 'is_flow': True, 'tags': ['GrossProfit']},
    'OperatingIncome':  {'unit': 'USD', 'is_flow': True, 'tags': ['OperatingIncomeLoss']},
    'NetIncome':        {'unit': 'USD', 'is_flow': True, 'tags': ['NetIncomeLoss', 'NetIncomeLossAvailableToCommonStockholdersBasic', 'ProfitLoss']},
    'DepreciationAmort':{'unit': 'USD', 'is_flow': True, 'tags': ['DepreciationDepletionAndAmortization', 'DepreciationAndAmortization', 'Depreciation', 'DepreciationAmortizationAndAccretionNet', 'DepreciationNonproduction']},
    'InterestExpense':  {'unit': 'USD', 'is_flow': True, 'tags': ['InterestExpense', 'InterestExpenseDebt', 'InterestPaidNet']},
    'ResearchDev':      {'unit': 'USD', 'is_flow': True, 'tags': ['ResearchAndDevelopmentExpense']},
    'OperatingExpenses':{'unit': 'USD', 'is_flow': True, 'tags': ['OperatingExpenses', 'CostsAndExpenses']},

    # --- Balance Sheet (instant, is_flow=False) ---
    'TotalAssets':       {'unit': 'USD', 'is_flow': False, 'tags': ['Assets']},
    'TotalLiabilities':  {'unit': 'USD', 'is_flow': False, 'tags': ['Liabilities', 'LiabilitiesCurrent']},
    'TotalEquity':       {'unit': 'USD', 'is_flow': False, 'tags': ['StockholdersEquity', 'StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest']},
    'Cash':              {'unit': 'USD', 'is_flow': False, 'tags': ['CashAndCashEquivalentsAtCarryingValue', 'CashCashEquivalentsAndShortTermInvestments', 'CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents', 'Cash', 'CashAndDueFromBanks']},
    'LongTermDebt':      {'unit': 'USD', 'is_flow': False, 'tags': ['LongTermDebtNoncurrent', 'LongTermDebt', 'LongTermDebtAndCapitalLeaseObligations', 'LongTermNotesPayable', 'SeniorLongTermNotes']},
    'CurrentDebt':       {'unit': 'USD', 'is_flow': False, 'tags': ['DebtCurrent', 'ShortTermBorrowings', 'LongTermDebtCurrent', 'LongTermDebtAndCapitalLeaseObligationsCurrent', 'CommercialPaper']},
    'Inventory':         {'unit': 'USD', 'is_flow': False, 'tags': ['InventoryNet', 'InventoryFinishedGoodsNetOfReserves']},
    'AccountsReceivable':{'unit': 'USD', 'is_flow': False, 'tags': ['AccountsReceivableNetCurrent', 'ReceivablesNetCurrent', 'AccountsNotesAndLoansReceivableNetCurrent', 'AccountsReceivableNet']},
    'AccountsPayable':   {'unit': 'USD', 'is_flow': False, 'tags': ['AccountsPayableCurrent', 'AccountsPayableTradeCurrent', 'AccountsPayableAndAccruedLiabilitiesCurrent']},
    'NetPPE':            {'unit': 'USD', 'is_flow': False, 'tags': ['PropertyPlantAndEquipmentNet', 'PublicUtilitiesPropertyPlantAndEquipmentNet', 'PropertyPlantAndEquipmentAndFinanceLeaseRightOfUseAssetAfterAccumulatedDepreciationAndAmortization']},
    'CurrentAssets':     {'unit': 'USD', 'is_flow': False, 'tags': ['AssetsCurrent']},
    'CurrentLiabilities':{'unit': 'USD', 'is_flow': False, 'tags': ['LiabilitiesCurrent']},

    # --- Shares (mixed: instant or duration depending on tag) ---
    'SharesOutstanding': {'unit': 'shares', 'is_flow': False, 'tags': ['CommonStockSharesOutstanding', 'EntityCommonStockSharesOutstanding', 'WeightedAverageNumberOfSharesOutstandingBasic']},
}

# Validate selected indicators
for ind in INDICATORS:
    if ind not in XBRL_CATALOG:
        raise ValueError(f"Unknown indicator '{ind}'. Available: {list(XBRL_CATALOG.keys())}")

# Compile frame-matching regex patterns
FRAME_PATTERNS = {
    ('annual',    'duration'): re.compile(r'^CY(\d{4})$'),
    ('annual',    'instant'):  re.compile(r'^CY(\d{4})Q4I$'),
    ('quarterly', 'duration'): re.compile(r'^CY(\d{4})Q(\d)$'),
    ('quarterly', 'instant'):  re.compile(r'^CY(\d{4})Q(\d)I$'),
}

HEADERS = {"User-Agent": f"BaThesisResearch {EMAIL}"}

# Identify which selected indicators are flows (for quarterly fill logic)
FLOW_INDICATORS = [ind for ind in INDICATORS if XBRL_CATALOG[ind]['is_flow']]
STOCK_INDICATORS = [ind for ind in INDICATORS if not XBRL_CATALOG[ind]['is_flow']]

print(f"Catalog loaded: {len(XBRL_CATALOG)} indicators available.")
print(f"Selected: {INDICATORS}")
print(f"  Flow items (income stmt): {FLOW_INDICATORS}")
print(f"  Stock items (balance sheet / shares): {STOCK_INDICATORS}")

Catalog loaded: 22 indicators available.
Selected: ['Revenue', 'COGS', 'GrossProfit', 'OperatingIncome', 'NetIncome', 'DepreciationAmort', 'InterestExpense', 'ResearchDev', 'OperatingExpenses', 'TotalAssets', 'TotalLiabilities', 'TotalEquity', 'Cash', 'LongTermDebt', 'CurrentDebt', 'Inventory', 'AccountsReceivable', 'AccountsPayable', 'NetPPE', 'CurrentAssets', 'CurrentLiabilities', 'SharesOutstanding']
  Flow items (income stmt): ['Revenue', 'COGS', 'GrossProfit', 'OperatingIncome', 'NetIncome', 'DepreciationAmort', 'InterestExpense', 'ResearchDev', 'OperatingExpenses']
  Stock items (balance sheet / shares): ['TotalAssets', 'TotalLiabilities', 'TotalEquity', 'Cash', 'LongTermDebt', 'CurrentDebt', 'Inventory', 'AccountsReceivable', 'AccountsPayable', 'NetPPE', 'CurrentAssets', 'CurrentLiabilities', 'SharesOutstanding']


In [ ]:
# Cell 4: Map tickers to CIK codes
url = "https://www.sec.gov/files/company_tickers.json"
resp = requests.get(url, headers=HEADERS)
resp.raise_for_status()

ticker_to_cik = {}
for entry in resp.json().values():
    ticker_to_cik[entry['ticker'].upper()] = str(entry['cik_str']).zfill(10)

found = {}
missing = []
for t in TICKERS:
    cik = ticker_to_cik.get(t.upper())
    if cik:
        found[t] = cik
    else:
        missing.append(t)

print(f"Mapped {len(found)}/{len(TICKERS)} tickers to CIK codes.")
if missing:
    print(f"\nNOT FOUND (not in SEC database): {missing}")
    print("These tickers will be skipped.")

Mapped 500/500 tickers to CIK codes.


In [ ]:
# Cell 5: Fetch data from EDGAR
#
# One API call per company. Both annual and quarterly data are extracted
# from the same response. SEC rate limit: 10 requests/second.
# For 500 companies this takes ~1-2 minutes.

def extract_item(facts_json, indicator_name, frequency, year_start, year_end):
    """
    Extract one indicator at one frequency from a company's EDGAR facts.
    Tries all XBRL tags and both frame patterns (duration + instant)
    to handle mixed-type indicators like SharesOutstanding.
    """
    config = XBRL_CATALOG[indicator_name]
    patterns = [FRAME_PATTERNS[(frequency, 'duration')],
                FRAME_PATTERNS[(frequency, 'instant')]]

    seen = set()
    rows = []

    for tag in config['tags']:
        for section in ['us-gaap', 'dei']:
            observations = (facts_json.get('facts', {})
                            .get(section, {})
                            .get(tag, {})
                            .get('units', {})
                            .get(config['unit'], []))

            for obs in observations:
                frame = obs.get('frame', '')
                matched_year = None
                matched_quarter = None
                for pat in patterns:
                    m = pat.match(frame)
                    if m:
                        matched_year = int(m.group(1))
                        if m.lastindex and m.lastindex >= 2:
                            matched_quarter = int(m.group(2))
                        break
                if matched_year is None:
                    continue
                if matched_year < year_start or matched_year > year_end:
                    continue
                dedup_key = (matched_year, matched_quarter)
                if dedup_key in seen:
                    continue
                seen.add(dedup_key)

                row = {'year': matched_year, 'end': obs.get('end'),
                       indicator_name: obs.get('val'),
                       f'_tag_{indicator_name}': tag}
                if frequency == 'quarterly':
                    row['quarter'] = matched_quarter
                rows.append(row)
    return rows


def build_panel(facts_json, frequency, year_start, year_end):
    """
    Extract all selected indicators at one frequency and merge
    into a single DataFrame for one company.
    """
    merge_keys = ['year'] if frequency == 'annual' else ['year', 'quarter']
    merged = None
    tags_used = {}

    for ind in INDICATORS:
        rows = extract_item(facts_json, ind, frequency, year_start, year_end)
        if not rows:
            continue
        df_item = pd.DataFrame(rows)
        tag_col = f'_tag_{ind}'
        tags_used[ind] = df_item[tag_col].unique().tolist()
        df_item = df_item.drop(columns=[tag_col])

        if merged is None:
            merged = df_item
        else:
            merged = merged.merge(
                df_item.drop(columns='end', errors='ignore'),
                on=merge_keys, how='outer'
            )
    return merged, tags_used


def fetch_with_retry(url, headers, max_retries=3):
    """Fetch URL with retry on rate-limit (429) or server error (5xx)."""
    for attempt in range(max_retries):
        resp = requests.get(url, headers=headers)
        if resp.status_code == 200:
            return resp.json()
        if resp.status_code == 429 or resp.status_code >= 500:
            wait = 10 * (attempt + 1)
            print(f"\n  [{resp.status_code}] Waiting {wait}s...", end=" ")
            time.sleep(wait)
            continue
        resp.raise_for_status()
    raise Exception(f"Failed after {max_retries} retries: {url}")


# --- Main fetch loop ---
all_annual = []
all_quarterly = []
errors = []
total = len(found)

print(f"Fetching {total} companies (annual + quarterly)...\n")

for i, (ticker, cik) in enumerate(found.items(), 1):
    if total > 20:
        if i % 50 == 0 or i == total:
            print(f"  [{i}/{total}]")
    else:
        print(f"  {ticker}", end="")

    try:
        url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
        facts = fetch_with_retry(url, HEADERS)
        entity_name = facts.get('entityName', ticker)
    except Exception as e:
        errors.append((ticker, str(e)))
        if total <= 20: print(f" FAILED")
        time.sleep(0.15)
        continue

    # Extract annual
    ann, ann_tags = build_panel(facts, 'annual', YEAR_START, YEAR_END)
    # Extract quarterly
    qtr, qtr_tags = build_panel(facts, 'quarterly', YEAR_START, YEAR_END)

    if ann is not None:
        ann['Ticker'] = ticker
        ann['Company'] = entity_name
        all_annual.append(ann)

    if qtr is not None:
        qtr['Ticker'] = ticker
        qtr['Company'] = entity_name
        all_quarterly.append(qtr)

    if total <= 20:
        a_n = len(ann) if ann is not None else 0
        q_n = len(qtr) if qtr is not None else 0
        all_tags = {**ann_tags, **qtr_tags}
        tag_info = ", ".join(f"{k}={'|'.join(v)}" for k, v in all_tags.items())
        print(f" OK (A:{a_n}, Q:{q_n}) — {tag_info}")

    time.sleep(0.12)


# --- Combine ---
def assemble_panel(parts, id_cols):
    if not parts:
        return pd.DataFrame()
    panel = pd.concat(parts, ignore_index=True)
    panel = panel.sort_values(id_cols).reset_index(drop=True)
    # Reorder: identifiers first, then indicators
    other = [c for c in panel.columns if c not in id_cols]
    return panel[id_cols + other]

panel_annual = assemble_panel(all_annual, ['Ticker', 'Company', 'year', 'end'])
panel_quarterly = assemble_panel(all_quarterly, ['Ticker', 'Company', 'year', 'quarter', 'end'])

print(f"\n{'='*60}")
if not panel_annual.empty:
    print(f"Annual:    {len(panel_annual)} obs, {panel_annual['Ticker'].nunique()} firms")
if not panel_quarterly.empty:
    print(f"Quarterly: {len(panel_quarterly)} obs, {panel_quarterly['Ticker'].nunique()} firms")

if errors:
    print(f"\nErrors ({len(errors)}):")
    for t, e in errors:
        print(f"  {t}: {e}")

Fetching 500 companies (annual + quarterly)...

  [50/500]
  [100/500]
  [150/500]
  [200/500]
  [250/500]
  [300/500]
  [350/500]
  [400/500]
  [450/500]
  [500/500]

Annual:    5379 obs, 500 firms
Quarterly: 21248 obs, 500 firms


In [ ]:
# Cell 6: Fill missing quarterly data
#
# Two types of fills:
#
# 1) FLOW ITEMS (income statement): the fiscal year-end quarter is missing
#    from EDGAR's quarterly frames. Since flows are additive (Annual = Q1+Q2+Q3+Q4),
#    we compute: Q_missing = Annual - sum(other 3 Qs).
#
# 2) STOCK ITEMS (balance sheet / shares): if a quarterly value is missing
#    but an annual value exists for that year, we use the annual value as
#    a fallback. This is reasonable for slowly-changing items like shares
#    outstanding or total assets.

if not panel_quarterly.empty and not panel_annual.empty:
    filled_flow = 0
    filled_stock = 0

    for ticker in panel_quarterly['Ticker'].unique():
        qtr_data = panel_quarterly[panel_quarterly['Ticker'] == ticker]
        ann_data = panel_annual[panel_annual['Ticker'] == ticker]

        for year in qtr_data['year'].unique():
            year_mask = (panel_quarterly['Ticker'] == ticker) & (panel_quarterly['year'] == year)
            year_quarters = panel_quarterly[year_mask]
            ann_year = ann_data[ann_data['year'] == year]
            if ann_year.empty:
                continue

            # --- Pass 1: Flow items (Annual - other 3 Qs) ---
            for ind in FLOW_INDICATORS:
                if ind not in panel_quarterly.columns or ind not in panel_annual.columns:
                    continue
                has_val = year_quarters[year_quarters[ind].notna()]
                missing_val = year_quarters[year_quarters[ind].isna()]
                if len(has_val) != 3 or len(missing_val) != 1:
                    continue
                annual_val = ann_year.iloc[0].get(ind)
                if pd.isna(annual_val):
                    continue
                fill_val = annual_val - has_val[ind].sum()
                panel_quarterly.at[missing_val.index[0], ind] = fill_val
                filled_flow += 1

            # --- Pass 2: Stock items (annual fallback) ---
            for ind in STOCK_INDICATORS:
                if ind not in panel_quarterly.columns or ind not in panel_annual.columns:
                    continue
                annual_val = ann_year.iloc[0].get(ind)
                if pd.isna(annual_val):
                    continue
                missing_mask = year_mask & panel_quarterly[ind].isna()
                n_missing = missing_mask.sum()
                if n_missing > 0:
                    panel_quarterly.loc[missing_mask, ind] = annual_val
                    filled_stock += n_missing

    # Re-sort
    panel_quarterly = panel_quarterly.sort_values(
        ['Ticker', 'year', 'quarter']
    ).reset_index(drop=True)

    # Reorder columns
    id_cols_q = ['Ticker', 'Company', 'year', 'quarter', 'end']
    other_cols_q = [c for c in panel_quarterly.columns if c not in id_cols_q]
    panel_quarterly = panel_quarterly[id_cols_q + other_cols_q]

    print(f"Filled {filled_flow} quarterly flow values (Annual - other 3 Qs).")
    print(f"Filled {filled_stock} quarterly stock values (annual fallback).")
else:
    print("No quarterly data to fill.")

Filled 22121 quarterly flow values (Annual - other 3 Qs).
Filled 5448 quarterly stock values (annual fallback).


In [ ]:
# Cell 7: Inspect the data

for label, panel in [('ANNUAL', panel_annual), ('QUARTERLY', panel_quarterly)]:
    if panel.empty:
        print(f"\n{label}: no data")
        continue
    print(f"\n{'='*60}")
    print(f"{label}: {len(panel)} obs, {panel['Ticker'].nunique()} firms")

    print(f"\nObservations per firm:")
    print(panel.groupby('Ticker').size().to_string())

    ind_cols = [c for c in INDICATORS if c in panel.columns]
    print(f"\nMissing values per firm × indicator:")
    missing_report = panel.groupby('Ticker')[ind_cols].apply(lambda g: g.isnull().sum())
    print(missing_report.to_string())

    print(f"\nSample rows:")
    display(panel.head(12))


ANNUAL: 5379 obs, 500 firms

Observations per firm:
Ticker
A        11
AAPL     11
ABBV     11
ABNB      8
ABT      11
ACGL     11
ACN      11
ADBE     11
ADI      11
ADM      11
ADP      11
ADSK     11
AEE      11
AEP      11
AES      11
AFL      11
AIG      11
AIZ      11
AJG      11
AKAM     11
ALB      11
ALGN     11
ALL      11
ALLE     11
AMAT     11
AMCR      9
AMD      11
AME      11
AMGN     11
AMP      11
AMT      11
AMZN     11
ANET     11
AON      11
AOS      11
APA       8
APD      11
APH      11
APO       7
APP       8
APTV     11
ARE      11
ARES     11
ATO      11
AVB      11
AVGO     11
AVY      11
AWK      11
AXON     11
AXP      11
AZO      11
BA       11
BAC      11
BALL     11
BAX      11
BBY      11
BDX      11
BEN      11
BF-B     11
BG        6
BIIB     11
BK       11
BKNG     11
BKR      11
BLDR     11
BLK       5
BMY      11
BR       11
BRK-B    11
BRO      11
BSX      11
BX       11
BXP      11
C        11
CAG      11
CAH      11
CARR      9
CAT      11
CB  

,Ticker,Company,year,end,Revenue,COGS,OperatingIncome,NetIncome,DepreciationAmort,InterestExpense,...,LongTermDebt,CurrentDebt,Inventory,AccountsReceivable,AccountsPayable,NetPPE,CurrentAssets,CurrentLiabilities,SharesOutstanding,GrossProfit
0,A,"AGILENT TECHNOLOGIES, INC.",2015,2015-10-31,4.038000e+09,1.997000e+09,5.220000e+08,4.010000e+08,2.530000e+08,66000000.0,...,1.653000e+09,8.000000e+07,5.540000e+08,6.170000e+08,2.500000e+08,5.940000e+08,3.399000e+09,9.470000e+08,6.120000e+08,NaN
1,A,"AGILENT TECHNOLOGIES, INC.",2016,2016-10-31,4.202000e+09,2.005000e+09,6.150000e+08,4.620000e+08,2.460000e+08,72000000.0,...,1.802000e+09,1.900000e+08,5.510000e+08,6.530000e+08,2.680000e+08,6.530000e+08,3.635000e+09,1.089000e+09,3.220000e+08,NaN
2,A,"AGILENT TECHNOLOGIES, INC.",2017,2017-10-31,4.472000e+09,2.073000e+09,8.070000e+08,6.840000e+08,2.120000e+08,79000000.0,...,1.800000e+09,3.450000e+08,6.080000e+08,7.510000e+08,2.920000e+08,7.920000e+08,4.397000e+09,1.361000e+09,3.230000e+08,NaN
3,A,"AGILENT TECHNOLOGIES, INC.",2018,2018-10-31,4.914000e+09,2.234000e+09,9.040000e+08,3.160000e+08,2.100000e+08,75000000.0,...,1.798000e+09,NaN,6.530000e+08,8.330000e+08,3.150000e+08,8.290000e+08,3.712000e+09,1.095000e+09,3.180000e+08,NaN
4,A,"AGILENT TECHNOLOGIES, INC.",2019,2019-10-31,5.163000e+09,2.358000e+09,9.410000e+08,1.071000e+09,2.380000e+08,74000000.0,...,1.787000e+09,6.750000e+08,7.060000e+08,9.660000e+08,3.290000e+08,8.440000e+08,3.102000e+09,1.892000e+09,3.100000e+08,NaN
5,A,"AGILENT TECHNOLOGIES, INC.",2020,2020-10-31,5.339000e+09,2.502000e+09,8.460000e+08,7.190000e+08,3.080000e+08,78000000.0,...,2.185000e+09,3.140000e+08,7.550000e+08,1.087000e+09,3.980000e+08,8.660000e+08,3.483000e+09,1.687000e+09,3.050000e+08,NaN
6,A,"AGILENT TECHNOLOGIES, INC.",2021,2021-10-31,6.319000e+09,2.912000e+09,1.347000e+09,1.210000e+09,3.210000e+08,81000000.0,...,2.730000e+09,NaN,8.790000e+08,1.205000e+09,4.750000e+08,9.740000e+08,3.474000e+09,1.584000e+09,3.000000e+08,NaN
7,A,"AGILENT TECHNOLOGIES, INC.",2022,2022-10-31,6.848000e+09,3.126000e+09,1.618000e+09,1.254000e+09,3.170000e+08,84000000.0,...,2.733000e+09,2.380000e+08,1.111000e+09,1.459000e+09,5.400000e+08,1.147000e+09,4.078000e+09,1.936000e+09,2.960000e+08,NaN
8,A,"AGILENT TECHNOLOGIES, INC.",2023,2023-10-31,6.833000e+09,3.368000e+09,1.350000e+09,1.240000e+09,2.710000e+08,95000000.0,...,2.555000e+09,NaN,1.033000e+09,1.295000e+09,4.880000e+08,1.314000e+09,4.338000e+09,1.617000e+09,2.930418e+08,NaN
9,A,"AGILENT TECHNOLOGIES, INC.",2024,2024-10-31,6.510000e+09,2.975000e+09,1.488000e+09,1.289000e+09,2.570000e+08,96000000.0,...,3.347000e+09,1.600000e+07,9.970000e+08,1.328000e+09,5.470000e+08,1.816000e+09,4.107000e+09,1.869000e+09,2.852322e+08,NaN



QUARTERLY: 21248 obs, 500 firms

Observations per firm:
Ticker
A        44
AAPL     44
ABBV     44
ABNB     26
ABT      44
ACGL     44
ACN      44
ADBE     44
ADI      44
ADM      44
ADP      44
ADSK     44
AEE      44
AEP      44
AES      44
AFL      44
AIG      44
AIZ      44
AJG      44
AKAM     44
ALB      44
ALGN     44
ALL      44
ALLE     44
AMAT     44
AMCR     36
AMD      44
AME      44
AMGN     44
AMP      44
AMT      44
AMZN     44
ANET     44
AON      44
AOS      44
APA      26
APD      44
APH      44
APO      22
APP      26
APTV     44
ARE      44
ARES     44
ATO      44
AVB      44
AVGO     40
AVY      44
AWK      44
AXON     44
AXP      44
AZO      44
BA       44
BAC      44
BALL     44
BAX      44
BBY      44
BDX      44
BEN      44
BF-B     44
BG       18
BIIB     44
BK       44
BKNG     44
BKR      41
BLDR     44
BLK      13
BMY      44
BR       44
BRK-B    44
BRO      44
BSX      44
BX       44
BXP      44
C        44
CAG      44
CAH      44
CARR     30
CAT      44


,Ticker,Company,year,quarter,end,Revenue,COGS,OperatingIncome,NetIncome,DepreciationAmort,...,CurrentDebt,Inventory,AccountsReceivable,AccountsPayable,NetPPE,CurrentAssets,CurrentLiabilities,SharesOutstanding,GrossProfit,ResearchDev
0,A,"AGILENT TECHNOLOGIES, INC.",2015,1,2015-04-30,9.630000e+08,483000000.0,107000000.0,87000000.0,NaN,...,80000000.0,556000000.0,576000000.0,261000000.0,593000000.0,3.620000e+09,9.300000e+08,610000000.0,NaN,81000000.0
1,A,"AGILENT TECHNOLOGIES, INC.",2015,2,2015-07-31,1.014000e+09,501000000.0,144000000.0,111000000.0,NaN,...,80000000.0,545000000.0,584000000.0,248000000.0,587000000.0,3.478000e+09,8.530000e+08,611000000.0,NaN,79000000.0
2,A,"AGILENT TECHNOLOGIES, INC.",2015,3,NaN,1.033000e+09,522000000.0,116000000.0,82000000.0,NaN,...,0.0,541000000.0,606000000.0,279000000.0,604000000.0,3.686000e+09,9.760000e+08,611000000.0,NaN,92000000.0
3,A,"AGILENT TECHNOLOGIES, INC.",2015,4,2016-01-31,1.028000e+09,491000000.0,155000000.0,121000000.0,66000000.0,...,80000000.0,554000000.0,617000000.0,250000000.0,594000000.0,3.399000e+09,9.470000e+08,612000000.0,NaN,78000000.0
4,A,"AGILENT TECHNOLOGIES, INC.",2016,1,2016-04-30,1.019000e+09,489000000.0,131000000.0,91000000.0,NaN,...,235000000.0,555000000.0,602000000.0,220000000.0,610000000.0,3.488000e+09,1.133000e+09,612000000.0,NaN,81000000.0
5,A,"AGILENT TECHNOLOGIES, INC.",2016,2,2016-07-31,1.044000e+09,502000000.0,146000000.0,124000000.0,NaN,...,235000000.0,543000000.0,590000000.0,261000000.0,623000000.0,3.530000e+09,1.151000e+09,613000000.0,NaN,86000000.0
6,A,"AGILENT TECHNOLOGIES, INC.",2016,3,NaN,1.072000e+09,521000000.0,132000000.0,79000000.0,NaN,...,0.0,533000000.0,631000000.0,257000000.0,639000000.0,3.635000e+09,9.450000e+08,614000000.0,NaN,83000000.0
7,A,"AGILENT TECHNOLOGIES, INC.",2016,4,2017-01-31,1.067000e+09,493000000.0,206000000.0,168000000.0,55000000.0,...,190000000.0,551000000.0,653000000.0,268000000.0,653000000.0,3.635000e+09,1.089000e+09,322000000.0,NaN,79000000.0
8,A,"AGILENT TECHNOLOGIES, INC.",2017,1,2017-04-30,1.102000e+09,510000000.0,201000000.0,164000000.0,NaN,...,241000000.0,548000000.0,677000000.0,265000000.0,675000000.0,3.800000e+09,1.187000e+09,321000000.0,NaN,84000000.0
9,A,"AGILENT TECHNOLOGIES, INC.",2017,2,2017-07-31,1.114000e+09,518000000.0,201000000.0,175000000.0,NaN,...,280000000.0,566000000.0,678000000.0,289000000.0,716000000.0,3.996000e+09,1.241000e+09,322000000.0,NaN,87000000.0


In [ ]:
# Cell 8: Download to Excel (two tabs)
from google.colab import files

file_name = 'edgar_panel_data.xlsx'
with pd.ExcelWriter(file_name) as writer:
    if not panel_annual.empty:
        panel_annual.to_excel(writer, sheet_name='Annual', index=False)
    if not panel_quarterly.empty:
        panel_quarterly.to_excel(writer, sheet_name='Quarterly', index=False)

files.download(file_name)
print(f"Downloaded: {file_name}")
print(f"  Annual tab:    {len(panel_annual)} rows")
print(f"  Quarterly tab: {len(panel_quarterly)} rows")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: edgar_panel_data.xlsx
  Annual tab:    5379 rows
  Quarterly tab: 21248 rows
